# Step 4: Explainability (XAI) with SHAP

**Goal:** demonstrate that the model learns genuine pathophysiology
rather than merely "memorizing" the data. The original paper
`chicco2020machine` shows that **Serum Creatinine** and **Ejection
Fraction** are the two most decisive indicators of survival for heart
failure patients — check whether the Extra Trees model (Step 3) "learned"
this correctly.

In [1]:
import sys
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

try:
    import imblearn  # noqa: F401
except ImportError:
    import subprocess, sys as _sys
    print("[i] imbalanced-learn not found in this environment (Colab/Kaggle) — installing...")
    subprocess.run([_sys.executable, '-m', 'pip', 'install', '-q', 'imbalanced-learn'], check=True)
try:
    import shap  # noqa: F401
except ImportError:
    import subprocess, sys as _sys
    print("[i] shap not found in this environment (Colab/Kaggle) — installing...")
    subprocess.run([_sys.executable, '-m', 'pip', 'install', '-q', 'shap'], check=True)

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import ExtraTreesClassifier
from imblearn.over_sampling import SMOTE
import shap

def _load_heart_failure_data():
    """Load the local file (../data/...) if present (running inside a
    cloned HMYT repo); otherwise (opened standalone via Colab/Kaggle, no
    accompanying data/ folder) automatically download it from the public
    mirror on hmyt-book (Public repo, verified 2026-09-23)."""
    import os
    local_path = "../data/heart_failure_clinical_records_dataset.csv"
    remote_url = ("https://raw.githubusercontent.com/fossbk-spec/hmyt-book/gh-pages/"
                  "labs_chuyen_de/ch02_suy_tim_risk_dxai/data/"
                  "heart_failure_clinical_records_dataset.csv")
    path = local_path if os.path.exists(local_path) else remote_url
    if path == remote_url:
        print(f"[i] Local data not found — downloading from the public mirror:\n    {remote_url}")
    return pd.read_csv(path).rename(columns={'death_event': 'DEATH_EVENT'})

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

df = _load_heart_failure_data()
FEATURE_COLS = [c for c in df.columns if c != 'DEATH_EVENT']
X, y = df[FEATURE_COLS], df['DEATH_EVENT'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

smote = SMOTE(random_state=RANDOM_STATE)
X_train_smote, y_train_smote = smote.fit_resample(X_train_s, y_train)

etc = ExtraTreesClassifier(n_estimators=300, random_state=RANDOM_STATE)
etc.fit(X_train_smote, y_train_smote)
print("[+] Retrained the exact Extra Trees + SMOTE model from Step 3.")


[+] Retrained the exact Extra Trees + SMOTE model from Step 3.


## 1. Computing SHAP Values with `TreeExplainer`

In [2]:
X_test_df = pd.DataFrame(X_test_s, columns=FEATURE_COLS)   # keep column names for readable plots

explainer = shap.TreeExplainer(etc)
shap_values = explainer.shap_values(X_test_df)

# the shap API may return a single array (positive class) or a 2-class list depending on version — normalize to the "Death" class (1)
if isinstance(shap_values, list):
    shap_vals_death = shap_values[1]
elif shap_values.ndim == 3:
    shap_vals_death = shap_values[:, :, 1]
else:
    shap_vals_death = shap_values

print(f"SHAP value matrix: {shap_vals_death.shape} (n_test={X_test_df.shape[0]}, n_features={X_test_df.shape[1]})")

SHAP value matrix: (60, 12) (n_test=60, n_features=12)


## 2. Overview Plot — `summary_plot`

In [3]:
plt.figure(figsize=(9, 6))
shap.summary_plot(shap_vals_death, X_test_df, show=False)
plt.title('SHAP Summary — Extra Trees + SMOTE (Death class)')
plt.tight_layout()
plt.savefig('../figures/04_shap_summary_en.png', dpi=110, bbox_inches='tight')
plt.close()
print("[+] Saved ../figures/04_shap_summary_en.png")

C:\Users\hoang\AppData\Local\Temp\ipykernel_36752\2049450484.py:2: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_vals_death, X_test_df, show=False)


[+] Saved ../figures/04_shap_summary_en.png


## 3. Clinical Comparison — Serum Creatinine & Ejection Fraction

In [4]:
mean_abs_shap = np.abs(shap_vals_death).mean(axis=0)
importance_df = pd.DataFrame({
    'feature': FEATURE_COLS,
    'mean_abs_shap': mean_abs_shap
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
importance_df.index = importance_df.index + 1  # rank starting at 1

print("Feature importance ranking by mean |SHAP| (REAL result from this run):")
print(importance_df.to_string())

top5 = set(importance_df['feature'].head(5))
for feat in ['serum_creatinine', 'ejection_fraction']:
    rank = importance_df.index[importance_df['feature'] == feat][0]
    status = "IS in the Top 5" if feat in top5 else "is NOT in the Top 5"
    print(f"\n-> {feat}: rank #{rank}/13 by mean |SHAP| — {status}")

Feature importance ranking by mean |SHAP| (REAL result from this run):
                     feature  mean_abs_shap
1                       time       0.155912
2          ejection_fraction       0.075912
3           serum_creatinine       0.068877
4                        age       0.030276
5               serum_sodium       0.028849
6        high_blood_pressure       0.027349
7                    anaemia       0.021023
8                   diabetes       0.014541
9   creatinine_phosphokinase       0.014301
10                       sex       0.013490
11                 platelets       0.011920
12                   smoking       0.010242

-> serum_creatinine: rank #3/13 by mean |SHAP| — IS in the Top 5

-> ejection_fraction: rank #2/13 by mean |SHAP| — IS in the Top 5


## 4. Comparative Discussion (fill in from the REAL result in the cell above)

> ⚠️ This section is a template — when finalizing the Step 5 report,
> replace each `[...]` with the actual numbers from the two cells above,
> from this exact run. Do NOT copy the fixed illustrative numbers.

- Rank of `serum_creatinine`: `[...]`/13. Rank of `ejection_fraction`:
  `[...]`/13.
- If BOTH are in the Top 5: the result **matches the literature**
  `chicco2020machine` — the model correctly learns the two most
  important physiological indicators (declining kidney function + the
  heart's pumping capacity), reinforcing the pipeline's clinical
  credibility.
- If NOT: consider whether **SMOTE distorted the original feature
  distribution** (SMOTE linearly interpolates between nearest neighbors
  in the scaled feature space, which can blur decision boundaries driven
  by 1-2 dominant features), or whether the small Test size (n=60) makes
  the SHAP estimate high-variance. This is a limitation to report
  honestly in the Step 5 report, not a bug to "fix" until it matches the
  literature at any cost.